#Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Install Packages

In [ ]:
# Install needed packages (usually already in Colab)
!pip install transformers pillow pandas torch --quiet

In [ ]:
# Import Libraries

In [ ]:
import pandas as pd
import torch
import requests
from PIL import Image
from io import BytesIO
from transformers import CLIPProcessor, CLIPModel

#Import Dataset

In [ ]:
# Load your dataset
input_path = "/content/drive/MyDrive/Venice_RC15/Datasets/Combined_Castello.csv"
df = pd.read_csv(input_path)

# Keep required columns only
df = df[['title', 'tags', 'url_s', 'one_comment']]

# Replace NaN with empty strings
df = df.fillna("")
df.head()


In [ ]:
#Importing model and processor

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


#Setting Categories

In [ ]:
categories = {
    "Culture & heritage": [
        "church", "cathedral", "historic", "heritage", "traditional",
        "culture", "museum", "monument", "statue", "saint", "art"
    ],
    "Commerce": [
        "shop", "market", "restaurant", "cafe", "store", "vendor", "commerce"
    ],
    "Tourism": [
        "tourist", "travel", "sightseeing", "landmark", "visit", "holiday"
    ],
    "Public space life": [
        "street", "people", "crowd", "sitting", "walking", "public", "social"
    ],
    "Water & mobility": [
        "canal", "boat", "gondola", "bridge", "vaporetto", "water", "lagoon"
    ],
    "Nature & environment": [
        "tree", "sky", "plants", "green", "nature", "water", "sunset"
    ],
    "Events": [
        "festival", "event", "carnival", "performance", "parade", "concert"
    ],
    "Architecture": [
        "building", "facade", "arch", "window", "bridge", "design", "structure"
    ]
}


#Categorization

In [ ]:
def classify_text(text):
    text = text.lower()
    labels = []

    for cat, keywords in categories.items():
        if any(k in text for k in keywords):
            labels.append(cat)

    return labels


In [ ]:
def classify_image(url):
    try:
        response = requests.get(url, timeout=10)
        image = Image.open(BytesIO(response.content)).convert("RGB")

        # Prepare model inputs
        inputs = processor(
            text=list(categories.keys()),
            images=image,
            return_tensors="pt",
            padding=True
        ).to(device)

        outputs = model(**inputs)
        logits = outputs.logits_per_image
        probs = logits.softmax(dim=1).cpu().detach().numpy()[0]

        # Pick labels with probability > threshold
        predicted_labels = [
            list(categories.keys())[i]
            for i, p in enumerate(probs) if p > 0.14
        ]

        return predicted_labels

    except:
        return []


In [ ]:
theme_results = []

for index, row in df.iterrows():
    combined_text = f"{row['title']} {row['tags']} {row['one_comment']}"
    text_labels = classify_text(combined_text)

    image_labels = classify_image(row['url_s'])

    # Combine + remove duplicates
    final_labels = list(set(text_labels + image_labels))

    theme_results.append(", ".join(final_labels))


#Export CSV

In [ ]:
df["theme_labels"] = theme_results

output_path = "/content/drive/MyDrive/Venice_RC15/Datasets/Combined_Castello_with_Themes.csv"
df.to_csv(output_path, index=False)

output_path
